# PrismaCoder-3B — Full Alignment Pipeline on Kaggle: SFT → DPO → GRPO (RL)

Trains **PrismaCoder-3B** (Qwen2.5-Coder-3B-Instruct base) through the *complete*
alignment stack from the research report — in one Kaggle session, top-to-bottom,
no manual edits:

| Stage | What it does | Method | Reward / Signal |
|---|---|---|---|
| **S1 · SFT** | Imitation of gate-green PrismaForge artifacts | QLoRA (r=16, α=32), completion-only loss | next-token on reference outputs |
| **S2 · DPO** | Preference optimization on verifier-labeled pairs | TRL `DPOTrainer`, β=0.05, lr 5e-7 | gate-differential pairs (chosen = gate-green, rejected = gate-failing) — including **on-policy pairs mined from the SFT model itself** |
| **S3 · GRPO** | **Online RL**: the model samples its own completions and learns from the verifier | TRL `GRPOTrainer`, group-relative advantages (no critic) | the V0–V5 verifier gate chain (`LightValidator` static gates on Kaggle; full Node gates in CI) |
| **S4 · Iterative RL** | Continual self-play: best-of-*k* sampling → fresh preference pairs → extra DPO round | RAFT-style loop | same verifier oracle |

GRPO **is** the RL stage of this pipeline (Group Relative Policy Optimization — the
PPO-family algorithm behind DeepSeek-R1): instead of a learned value model, each
prompt's *group* of `num_generations` completions is scored by the Prisma verifier
and advantages are computed relative to the group mean, so the reward is exactly the
same gate chain that filters the dataset and labels the DPO pairs.

**Data**: downloads **PrismaForge** from the HF Hub when available; otherwise builds a
deterministic mini-corpus with the repo's own synthetic generator (seeded, offline).
**Verification**: `LightValidator` (V0 static gates: PSL parse, fence balance, symbol
grounding) — no Node.js needed on Kaggle; records stay `unverified` pending a full-gate
re-run with `benchmark/evaluator/evaluate.py --full-gates`.

**Hardware budget (T4 ×2 or P100, 12 h session, defaults below)**:
SFT ≈ 4–6 h · pair mining ≈ 25 min · DPO ≈ 1.5 h · GRPO ≈ 2 h · iterative RL ≈ 45 min ·
stage evals ≈ 30 min · merges + GGUF ≈ 40 min. Every stage saves checkpoints and can be
skipped via the `RUN_*` flags in Cell 2.

> Attach the repo as a Kaggle Dataset (Add Data → upload `PrismaCoder.zip`) and the
> notebook picks up the package automatically; otherwise it clones `REPO_URL`.


In [ ]:
# Cell 1 - install the pinned alignment stack (SFT + DPO + GRPO)
%pip install -q -U transformers==4.52.3 trl==0.19.1 peft==0.16.0 \
    datasets==3.6.0 accelerate==1.8.1 bitsandbytes==0.46.1 \
    huggingface-hub==0.33.4 sentencepiece tensorboard matplotlib \
    'PyYAML>=6.0' requests

import importlib
for _pkg in ['transformers', 'trl', 'peft', 'datasets', 'accelerate', 'bitsandbytes']:
    print(_pkg, importlib.import_module(_pkg).__version__)

In [ ]:
# Cell 2 - configuration, stage switches, secrets
import os, torch

SEED = 20260830                      # report default seed
BASE_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
HF_USERNAME = 'prismacoder'          # change to your org when pushing
DATASET_REPO = f'{HF_USERNAME}/PrismaForge'
OUTPUT_REPO = f'{HF_USERNAME}/PrismaCoder-3B-RL'
REPO_URL = os.environ.get('PRISMACODER_REPO_URL', 'https://github.com/prismacoder/PrismaCoder.git')

# ---- stage switches: skip any stage without touching other cells -----------
RUN_SFT = True
RUN_DPO = True
RUN_GRPO = True                      # the RL stage
RUN_ITERATIVE_RL = True              # S4 self-play round on top of GRPO

# ---- SFT (report Table 12.1 S1: QLoRA r16/a32, lr 1e-4, cosine 3% warmup) --
MAX_TRAIN_SAMPLES = 4000             # 0 = all available
MAX_VAL_SAMPLES = 200
SFT_EPOCHS = 3
SFT_LR = 1e-4
MAX_SEQ_LEN = 4096

# ---- DPO (S2: gate-differential pairs, beta 0.05, lr 5e-7) -----------------
MAX_DPO_PAIRS = 1500
ONPOLICY_MINING_PROMPTS = 120        # prompts used to mine on-policy pairs
DPO_EPOCHS = 1
DPO_LR = 5e-7
DPO_BETA = 0.05

# ---- GRPO (S3: online RL with the verifier reward) -------------------------
GRPO_PROMPTS = 150
GRPO_NUM_GENERATIONS = 4
GRPO_MAX_COMPLETION = 640
GRPO_LR = 1e-6
GRPO_BETA = 0.04                     # KL coefficient against the (adapter-off) policy
GRPO_TEMPERATURE = 0.9

# ---- S4: iterative RL round (best-of-k self-play pairs) --------------------
RL2_PROMPTS = 40
RL2_K = 4
RL2_MIN_PAIRS = 8

EVAL_PROMPTS = 40                    # held-out prompts for stage-wise gate pass@1

WORK = '/kaggle/working'
DATA_DIR = f'{WORK}/data'
CKPT = f'{WORK}/checkpoints'
MERGED = f'{WORK}/merged'
STAGE_RESULTS = f'{WORK}/stage_results.json'
for _d in (DATA_DIR, CKPT, MERGED):
    os.makedirs(_d, exist_ok=True)

def _secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, '')

HF_TOKEN = _secret('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
print('HF token:', 'loaded' if HF_TOKEN else 'not set (public paths only, push skipped)')

IS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
TORCH_DTYPE = torch.bfloat16 if IS_BF16 else torch.float16
print('torch', torch.__version__, '| gpus:', torch.cuda.device_count(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
      '| dtype:', TORCH_DTYPE)

In [ ]:
# Cell 3 - locate / fetch the PrismaCoder repo (package code + verifier)
import glob, os, subprocess, sys, zipfile
from pathlib import Path

def find_repo_root() -> str | None:
    # (a) already-extracted dataset mounted under /kaggle/input
    for pat in ('/kaggle/input/*/prismacoder/__init__.py',
                '/kaggle/input/*/*/prismacoder/__init__.py'):
        hits = sorted(glob.glob(pat))
        if hits:
            return str(Path(hits[0]).parent.parent)
    # (b) PrismaCoder.zip uploaded as a Kaggle dataset -> extract to working
    for z in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                if any('prismacoder/__init__.py' in n for n in names):
                    dest = Path(WORK) / 'repo_src'
                    dest.mkdir(parents=True, exist_ok=True)
                    zf.extractall(dest)           # inside the with: zf must stay open
                    # tolerate archives that carry a top-level folder prefix
                    tops = {Path(n).parts[0] for n in names if n.strip()}
                    root_out = dest
                    if len(tops) == 1 and not (dest / 'prismacoder').exists():
                        root_out = dest / tops.pop()
                    print('extracted', z, '->', root_out)
                    return str(root_out)
        except zipfile.BadZipFile:
            continue
    return None

ROOT = find_repo_root()
if ROOT is None:
    print('repo not found under /kaggle/input - cloning', REPO_URL)
    ROOT = str(Path(WORK) / 'PrismaCoder')
    if not Path(ROOT, 'prismacoder', '__init__.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, ROOT], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from prismacoder import __version__ as pc_version
print('PrismaCoder package', pc_version, '| root:', ROOT)

In [ ]:
# Cell 4 - data bootstrap: PrismaForge from the HF Hub, else a deterministic
# mini-corpus built by the repo's own synthetic generator (fully offline).
import random
from pathlib import Path

from prismacoder.io_utils import read_jsonl, write_jsonl

train_file = Path(DATA_DIR) / 'train.jsonl'
val_file = Path(DATA_DIR) / 'validation.jsonl'
pairs_file = Path(DATA_DIR) / 'dpo_pairs.jsonl'

def _download_from_hub() -> bool:
    try:
        from huggingface_hub import hf_hub_download
        for fname, target in (('train.jsonl', train_file),
                              ('validation.jsonl', val_file)):
            hf_hub_download(DATASET_REPO, fname, repo_type='dataset',
                            local_dir=str(DATA_DIR))
            assert target.exists() and target.stat().st_size > 0
        return True
    except Exception as e:
        print('hub download unavailable:', type(e).__name__, '-', e)
        return False

if _download_from_hub():
    train_records = read_jsonl(str(train_file))
    val_records = read_jsonl(str(val_file))
    if pairs_file.exists():
        static_pairs = read_jsonl(str(pairs_file))
    else:
        try:
            from huggingface_hub import hf_hub_download
            hf_hub_download(DATASET_REPO, 'dpo_pairs.jsonl', repo_type='dataset',
                            local_dir=str(DATA_DIR))
            static_pairs = read_jsonl(str(pairs_file))
        except Exception:
            static_pairs = []
            print('no dpo_pairs.jsonl on the hub - on-policy mining will supply pairs')
    print('PrismaForge from hub:', len(train_records), 'train /', len(val_records), 'val /',
          len(static_pairs), 'DPO pairs')
else:
    print('building the deterministic mini-corpus (repo synthetic generator, seed', SEED, ')...')
    from prismacoder.synthetic.generator import SyntheticGenerator
    _n_target = max(MAX_TRAIN_SAMPLES or 4000, 1200) + 250
    _gen = SyntheticGenerator(SEED)
    recs = _gen.generate(_n_target)
    _gen.attach_reasoning(recs, fraction=0.10)
    _val_n = min(MAX_VAL_SAMPLES, max(40, len(recs) // 20))
    val_records = recs[-_val_n:]
    train_records = recs[:-_val_n]
    static_pairs = []
    for r in train_records:
        p = r.get('preference')
        if not p:
            continue
        static_pairs.append({
            'id': r['id'],
            'prompt': p['prompt'],
            'context_schema': r['task'].get('context', {}).get('schema', ''),
            'chosen': p['chosen'],
            'rejected': p['rejected'],
            'basis': p['basis'],
        })
    write_jsonl(str(train_file), train_records)
    write_jsonl(str(val_file), val_records)
    write_jsonl(str(pairs_file), static_pairs)
    print('mini-corpus:', len(train_records), 'train /', len(val_records), 'val /',
          len(static_pairs), 'gate-differential DPO pairs')

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]

# Deterministic, disjoint prompt pools sliced out of the TRAIN pool
#   [0 : GRPO_PROMPTS]                      -> GRPO (online RL)
#   [GRPO : +ONPOLICY_MINING_PROMPTS]       -> DPO on-policy pair mining
#   [GRPO+MINING : +RL2_PROMPTS]            -> iterative RL round
# The VALIDATION pool is never used for training or RL (held out for eval).
_rng = random.Random(SEED)
_train_idx = list(range(len(train_records)))
_rng.shuffle(_train_idx)
grpo_pool = [train_records[i] for i in _train_idx[:GRPO_PROMPTS]]
mine_pool = [train_records[i] for i in _train_idx[GRPO_PROMPTS:
                                                  GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS]]
rl2_pool = [train_records[i] for i in _train_idx[GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS:
                                                 GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS + RL2_PROMPTS]]
eval_pool = val_records[:EVAL_PROMPTS]
print('pools -> grpo:', len(grpo_pool), '| mining:', len(mine_pool),
      '| rl2:', len(rl2_pool), '| eval (held-out):', len(eval_pool))

In [ ]:
# Cell 5 - tokenizer + 4-bit base model + the shared verifier-gated scorers
import gc, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from prismacoder.constants import SYSTEM_PROMPT
from prismacoder.verification.verifier import LightValidator

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=TORCH_DTYPE,
    attn_implementation='sdpa',      # T4 has no flash-attn wheels
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

LIGHT = LightValidator()             # V0 static gates: PSL parse, fences, symbol grounding
RE_FENCE = re.compile(r'```(?:\w+)?\n(.*?)```', re.S)

def extract_artifact(text: str) -> str:
    """Pull the code payload out of a model completion (fenced block preferred)."""
    fences = RE_FENCE.findall(text or '')
    return '\n'.join(fences) if fences else (text or '')

def score_artifact(schema: str, artifact: str) -> tuple[float, str]:
    """Gate-aligned graded score. 1.0 = V0 pass; failure gets shaped partial
    credit (fence balance, model grounding) so RL gradients still flow."""
    text = extract_artifact(artifact)
    if not text.strip():
        return -0.2, 'empty'
    verdict = LIGHT.validate(schema, text)
    if verdict.status != 'failed':
        return 1.0, verdict.status
    diags = [d.lower() for d in verdict.diagnostics]
    s = 0.0
    if 'unbalanced code fences' not in diags and '```' in text:
        s += 0.15
    if not any(d.startswith('ungrounded model reference') for d in diags):
        s += 0.15
    if any('ungrounded field' in d for d in diags):
        s += 0.05
    return s, 'failed'

@torch.no_grad()
def generate_completion(messages: list[dict], *, temperature: float = 0.0,
                        max_new_tokens: int = 640, k: int = 1) -> list[str]:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)
    if temperature and temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=k)
    else:
        gen_kwargs.update(do_sample=False, num_return_sequences=k)
    out = model.generate(**inputs, **gen_kwargs)
    return [tokenizer.decode(o[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for o in out]

def messages_for(rec: dict) -> list[dict]:
    from training.trainer.data import build_user_message
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_message(rec)}]

def schema_of(rec: dict) -> str:
    return rec['task'].get('context', {}).get('schema', '')

@torch.no_grad()
def gate_pass_rate(records: list[dict], *, temperature: float = 0.0) -> float:
    """V0 gate pass@1 on held-out records - the stage-progress metric."""
    model.eval()
    passed = 0
    for rec in records:
        text = generate_completion(messages_for(rec), temperature=temperature)[0]
        score, status = score_artifact(schema_of(rec), text)
        passed += status not in ('failed', 'empty')
    return passed / max(1, len(records))

def load_stage_results() -> dict:
    import json
    p = Path(STAGE_RESULTS)
    if p.exists():
        return json.loads(p.read_text())
    return {}

def record_stage(name: str, value: float) -> None:
    import json
    res = load_stage_results()
    res[name] = round(value, 4)
    Path(STAGE_RESULTS).write_text(json.dumps(res, indent=2))
    print(f'[stage] {name}: gate pass@1 = {value:.3f}')

def free_memory(*objs) -> None:
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('base model ready on', next(model.parameters()).device)

In [ ]:
# Cell 6 - baseline: gate pass@1 of the *untuned* base model (the number to beat)
base_pass = gate_pass_rate(eval_pool)
record_stage('base', base_pass)

## Stage 1 — SFT: imitation of gate-green artifacts (report §12, S1)

Supervised fine-tuning with **completion-only loss**: the model sees
`system → user (instruction + schema context) → assistant (reference artifact)`
and the loss is masked to the assistant span. Hyperparameters follow report
Table 12.1: QLoRA r=16 / α=32 on all linear projections, lr 1e-4, cosine
schedule with 3% warmup, effective batch 16, `paged_adamw_8bit`. A plain
`transformers.Trainer` is used with a pre-tokenized dataset — version-stable
and equivalent to `SFTTrainer` for this format (the 10% of records carrying a
mechanical plan-span get it prepended, report §12.1).

Output: `checkpoints/sft_adapter` → merged to `merged/sft` (the DPO base).


In [ ]:
# Cell 8 - Stage 1a: tokenize PrismaForge into chat examples (completion-only loss)
from datasets import Dataset

from training.trainer.data import to_chat_example

def tokenize_record(rec: dict) -> dict:
    ex = to_chat_example(rec, tokenizer, with_reasoning=True)
    return {'input_ids': ex['input_ids'], 'labels': ex['labels']}

_sft_train = [tokenize_record(r) for r in train_records]
_sft_val = [tokenize_record(r) for r in val_records[:MAX_VAL_SAMPLES]] if val_records else []
train_ds = Dataset.from_list(_sft_train)
val_ds = Dataset.from_list(_sft_val) if _sft_val else None
print(train_ds)

In [ ]:
# Cell 9 - Stage 1b: QLoRA SFT (plain Trainer, completion-only loss)
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments
from torch.nn.utils.rnn import pad_sequence

LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

class LossMaskCollator:
    """Pad pre-tokenized examples; labels padded with -100 (ignored)."""
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, feats: list[dict]) -> dict:
        ids = [torch.tensor(f['input_ids']) for f in feats]
        lbs = [torch.tensor(f['labels']) for f in feats]
        ids = pad_sequence(ids, batch_first=True, padding_value=self.pad_id)
        lbs = pad_sequence(lbs, batch_first=True, padding_value=-100)
        return {'input_ids': ids, 'labels': lbs,
                'attention_mask': (ids != self.pad_id).long()}

if RUN_SFT:
    from training.trainer.data import SoftOrderingMixer

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={'use_reentrant': False}
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    model = get_peft_model(model, LoraConfig(
        task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
        bias='none', target_modules=LORA_TARGETS,
    ))
    model.print_trainable_parameters()

    mixer = SoftOrderingMixer(seed=SEED)
    ordered = list(mixer.stream(train_records, SFT_EPOCHS))
    epoch_ds = Dataset.from_list([tokenize_record(r) for r in ordered])

    args = TrainingArguments(
        output_dir=f'{CKPT}/sft_run',
        run_name='prismacoder-3b-sft-kaggle',
        seed=SEED,
        num_train_epochs=1,                       # the mixer already repeats 3 epochs
        learning_rate=SFT_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,            # effective batch 16
        bf16=IS_BF16,
        fp16=not IS_BF16,                         # T4: fp16 + GradScaler
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        weight_decay=0.01,
        logging_steps=20,
        eval_strategy='steps' if val_ds is not None else 'no',
        eval_steps=250,
        save_strategy='steps', save_steps=500, save_total_limit=2,
        report_to=['tensorboard'],
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )

    sft_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=epoch_ds,
        eval_dataset=val_ds,
        data_collator=LossMaskCollator(tokenizer.pad_token_id),
    )
    sft_trainer.train()
    sft_trainer.save_model(f'{CKPT}/sft_adapter')
    tokenizer.save_pretrained(f'{CKPT}/sft_adapter')
    free_memory(sft_trainer)
    print('SFT adapter ->', f'{CKPT}/sft_adapter')
else:
    print('RUN_SFT=False - skipping (reuse a previous session\'s adapter or merged model)')

In [ ]:
# Cell 10 - Stage 1c: merge the SFT adapter into the base weights -> merged/sft,
# then measure the SFT model's gate pass@1 on the held-out pool.
import shutil, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

SFT_MERGED = f'{MERGED}/sft'

if RUN_SFT:
    free_memory(model)
    base_cpu = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/sft_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(SFT_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(SFT_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged SFT ->', SFT_MERGED)

    # reload the merged SFT model in 4-bit for everything that follows
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('sft', gate_pass_rate(eval_pool))
else:
    SFT_MERGED = None
    print('skipped (RUN_SFT=False)')

## Stage 2 — DPO: preference optimization on verifier-labeled pairs (report §12, S2)

DPO needs *pairs* (chosen ≻ rejected). PrismaForge labels them with the same
verifier used everywhere else (**gate-differential basis**: chosen = gate-green
reference, rejected = the strongest gate-failing mutant). This stage goes one
step further and mines **on-policy pairs**: the SFT model itself samples two
completions per prompt, the V0 gate scores both, and the differential becomes a
fresh pair — the standard bridge from offline DPO to online RL.

- Static pairs: hub `dpo_pairs.jsonl`, or generator preference records.
- On-policy pairs: `ONPOLICY_MINING_PROMPTS` × 2 samples, kept only when the
  gate verdicts actually differ.
- Hyperparameters (Table 12.1): β=0.05, lr 5e-7, 1 epoch, effective batch 16,
  reference model = the SFT model (adapter-disable path — no second model in memory).

Output: `checkpoints/dpo_adapter` → merged to `merged/dpo` (the GRPO base).


In [ ]:
# Cell 12 - Stage 2a: build the DPO pair set (static gate-differential pairs
# + on-policy pairs mined from the SFT model through the V0 gate).
import random
from pathlib import Path

from prismacoder.io_utils import write_jsonl

final_pairs: list[dict] = [p for p in static_pairs if p.get('chosen') and p.get('rejected')]

if RUN_SFT and RUN_DPO and mine_pool:
    rng = random.Random(SEED + 1)
    mined = 0
    for rec in mine_pool:
        msgs = messages_for(rec)
        outs = generate_completion(msgs, temperature=0.8, max_new_tokens=512, k=2)
        scored = [(*score_artifact(schema_of(rec), t), t) for t in outs]
        scored.sort(key=lambda x: x[0], reverse=True)
        (hi_s, hi_st, hi_t), (lo_s, lo_st, lo_t) = scored[0], scored[-1]
        # keep the pair only when the gate actually separates the two samples
        if hi_s > lo_s and (hi_st != 'failed' or lo_st == 'failed' or hi_s - lo_s >= 0.2):
            final_pairs.append({
                'id': f"{rec['id']}-onpolicy",
                'prompt': rec['task']['instruction'],
                'context_schema': schema_of(rec),
                'chosen': extract_artifact(hi_t),
                'rejected': extract_artifact(lo_t),
                'basis': f'on-policy gate differential ({hi_st} vs {lo_st})',
            })
            mined += 1
    print(f'on-policy mining: {mined}/{len(mine_pool)} prompts produced separable pairs')

rng = random.Random(SEED + 2)
rng.shuffle(final_pairs)
final_pairs = final_pairs[:MAX_DPO_PAIRS]
DPO_PAIRS_FILE = str(Path(DATA_DIR) / 'dpo_pairs_final.jsonl')
write_jsonl(DPO_PAIRS_FILE, final_pairs)
print('DPO pair set:', len(final_pairs), '->', DPO_PAIRS_FILE)

In [ ]:
# Cell 13 - Stage 2b: DPO training (TRL DPOTrainer, ref = adapter-disabled SFT model)
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

from training.trainer.data import to_dpo_example

DPO_MERGED = f'{MERGED}/dpo'

if RUN_DPO:
    free_memory(model)
    dpo_base_dir = SFT_MERGED or BASE_MODEL
    model = AutoModelForCausalLM.from_pretrained(
        dpo_base_dir, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )

    pair_rows = read_jsonl(DPO_PAIRS_FILE)
    dpo_ds = Dataset.from_list([to_dpo_example(p, tokenizer) for p in pair_rows])
    print(dpo_ds)

    dpo_args = DPOConfig(
        output_dir=f'{CKPT}/dpo_run',
        run_name='prismacoder-3b-dpo-kaggle',
        seed=SEED,
        num_train_epochs=DPO_EPOCHS,
        learning_rate=DPO_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,            # effective batch 16
        bf16=IS_BF16,
        fp16=not IS_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        beta=DPO_BETA,                            # 0.05 per report Table 12.1
        loss_type='sigmoid',
        max_prompt_length=1536,
        max_completion_length=1024,
        logging_steps=10,
        save_strategy='steps', save_steps=400, save_total_limit=2,
        report_to=['tensorboard'],
    )

    dpo_trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=dpo_ds,
        processing_class=tokenizer,
        peft_config=LoraConfig(
            task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
            bias='none', target_modules=LORA_TARGETS,
        ),
    )
    dpo_trainer.train()
    dpo_trainer.save_model(f'{CKPT}/dpo_adapter')
    tokenizer.save_pretrained(f'{CKPT}/dpo_adapter')
    free_memory(dpo_trainer)
    print('DPO adapter ->', f'{CKPT}/dpo_adapter')
else:
    print('RUN_DPO=False - skipping')

In [ ]:
# Cell 14 - Stage 2c: merge the DPO adapter -> merged/dpo (the GRPO base),
# measure gate pass@1, then free merged/sft to stay inside the disk quota.
import os, shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM

if RUN_DPO:
    free_memory(model)
    dpo_base_dir = SFT_MERGED or BASE_MODEL
    base_cpu = AutoModelForCausalLM.from_pretrained(
        dpo_base_dir, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/dpo_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(DPO_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(DPO_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged DPO ->', DPO_MERGED)

    if SFT_MERGED and os.path.isdir(SFT_MERGED):
        shutil.rmtree(SFT_MERGED)          # merged/sft (~6 GB) no longer needed
        print('freed disk:', SFT_MERGED)

    model = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('dpo', gate_pass_rate(eval_pool))
else:
    DPO_MERGED = SFT_MERGED or BASE_MODEL
    print('skipped (RUN_DPO=False) - GRPO will start from', DPO_MERGED)

## Stage 3 — GRPO: online reinforcement learning with the verifier as reward (report §12, S3)

This is the **RL stage**. GRPO (Group Relative Policy Optimization, the PPO-family
algorithm behind DeepSeek-R1) removes the learned critic: for every prompt the policy
samples a *group* of `num_generations` completions, each completion is scored by the
**same Prisma verifier that filters the dataset and labels the DPO pairs**, advantages
are normalized within the group (`r - mean(r)) / std(r)`), and the policy gradient is
clipped like PPO, with an optional KL term (`beta`) against the frozen reference
(the DPO model with adapters disabled — no extra model in memory).

**Reward (gate-aligned, graded)** — implemented in Cell 17 with `LightValidator` (V0):
- `+1.0` V0 pass (PSL/TS parses, balanced fences, every referenced model/field is grounded)
- failure gets shaped partial credit: `+0.15` balanced fences, `+0.15` no ungrounded
  model references, `+0.05` mild field grounding; `-0.2` empty output — so gradients
  flow even when every sample in a group fails the hard gate.
On a Node-enabled machine the same function swaps in `Verifier.verify()` for the full
V1–V5 chain (`prisma validate/generate → tsc → db push → vitest`) — the reward *is* the
gate chain, which is the project's core idea (report §10–11).

Hyperparameters: lr 1e-6, `num_generations=4`, temperature 0.9, KL β=0.04,
`use_vllm=False` (transformers generation — T4-safe). Generation dominates the wall
clock; ~150 prompts ≈ 2 h on T4 ×2.

Output: `checkpoints/grpo_adapter` → merged to `merged/grpo`.


In [ ]:
# Cell 16 - Stage 3a: the verifier reward function + the GRPO prompt dataset
from datasets import Dataset

def prisma_verifier_reward(completions, schema_psl=None, **kwargs):
    """GRPO reward = the Prisma verifier. Extra dataset columns (here
    `schema_psl`) are forwarded by GRPOTrainer as kwargs lists."""
    schemas = schema_psl or [''] * len(completions)
    rewards = []
    for completion, schema in zip(completions, schemas):
        text = completion[0]['content'] if isinstance(completion, list) else completion
        score, _status = score_artifact(schema, text)
        rewards.append(score)
    return rewards

grpo_rows = []
for rec in grpo_pool:
    if not schema_of(rec).strip():
        continue
    grpo_rows.append({
        'prompt': messages_for(rec),          # conversational prompt (TRL applies the template)
        'schema_psl': schema_of(rec),         # extra column -> reward kwarg
        'task_id': rec['id'],
    })
grpo_ds = Dataset.from_list(grpo_rows)
print(grpo_ds)
print('example reward check:',
      prisma_verifier_reward([grpo_rows[0]['prompt'][1]['content']], [grpo_rows[0]['schema_psl']]))

In [ ]:
# Cell 17 - Stage 3b: GRPO training (TRL GRPOTrainer + QLoRA, verifier reward)
import torch
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

GRPO_MERGED = f'{MERGED}/grpo'

if RUN_GRPO:
    free_memory(model)
    model = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = True            # generation-heavy stage: KV cache ON
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )

    grpo_args = GRPOConfig(
        output_dir=f'{CKPT}/grpo_run',
        run_name='prismacoder-3b-grpo-kaggle',
        seed=SEED,
        num_train_epochs=1,
        learning_rate=GRPO_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,        # 2*1*4 = 8 completions/step = 2 groups of 4
        num_generations=GRPO_NUM_GENERATIONS, # group size for relative advantages
        max_prompt_length=1536,
        max_completion_length=GRPO_MAX_COMPLETION,
        temperature=GRPO_TEMPERATURE,
        beta=GRPO_BETA,                       # KL vs adapter-disabled DPO policy
        use_vllm=False,                       # transformers generation (T4-safe)
        bf16=IS_BF16,
        fp16=not IS_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        logging_steps=5,
        save_strategy='no',
        report_to=['tensorboard'],
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=[prisma_verifier_reward],
        args=grpo_args,
        train_dataset=grpo_ds,
        processing_class=tokenizer,
        peft_config=LoraConfig(
            task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
            bias='none', target_modules=LORA_TARGETS,
        ),
    )
    grpo_trainer.train()
    grpo_trainer.save_model(f'{CKPT}/grpo_adapter')
    tokenizer.save_pretrained(f'{CKPT}/grpo_adapter')
    free_memory(grpo_trainer)
    print('GRPO adapter ->', f'{CKPT}/grpo_adapter')
else:
    print('RUN_GRPO=False - skipping')

In [ ]:
# Cell 18 - Stage 3c: merge the GRPO adapter -> merged/grpo, measure pass@1,
# then free merged/dpo to stay inside the disk quota.
import os, shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM

if RUN_GRPO:
    free_memory(model)
    base_cpu = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/grpo_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(GRPO_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(GRPO_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged GRPO ->', GRPO_MERGED)

    if RUN_DPO and os.path.isdir(DPO_MERGED) and DPO_MERGED != BASE_MODEL:
        shutil.rmtree(DPO_MERGED)
        print('freed disk:', DPO_MERGED)

    model = AutoModelForCausalLM.from_pretrained(
        GRPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('grpo', gate_pass_rate(eval_pool))
else:
    GRPO_MERGED = DPO_MERGED
    print('skipped (RUN_GRPO=False)')

## Stage 4 — Iterative RL: the continual self-play loop (report §12.4, optional)

One GRPO pass is a single RL iteration; production RLHF runs the loop **repeatedly**:
sample *k* completions per fresh prompt from the current policy → let the verifier pick
winners/losers → distill the differential into a short DPO round → merge → repeat. This
cell runs **one extra iteration** (RAFT-style best-of-*k* self-play) on top of the GRPO
model, using prompts disjoint from everything seen so far. Set `RUN_ITERATIVE_RL=False`
in Cell 2 to skip, or duplicate this cell (with a fresh prompt slice) for more rounds.

Output (when it fires): `checkpoints/rl2_adapter` → merged to `merged/rl2` = **the final model**.


In [ ]:
# Cell 20 - Stage 4: iterative RL round - best-of-k self-play pairs -> short DPO
import os, shutil
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

RL2_MERGED = f'{MERGED}/rl2'
FINAL_DIR = GRPO_MERGED

if RUN_ITERATIVE_RL and rl2_pool:
    # 4a. sample k completions per fresh prompt from the current policy
    selfplay_pairs = []
    for rec in rl2_pool:
        schema = schema_of(rec)
        if not schema.strip():
            continue
        outs = generate_completion(messages_for(rec), temperature=1.0,
                                   max_new_tokens=512, k=RL2_K)
        scored = sorted(((score_artifact(schema, t)[0], t) for t in outs),
                        key=lambda x: x[0], reverse=True)
        best_s, best_t = scored[0]
        worst_s, worst_t = scored[-1]
        if best_s >= 0.8 and best_s > worst_s:      # winner must clear the gate
            selfplay_pairs.append({
                'prompt': rec['task']['instruction'],
                'context_schema': schema,
                'chosen': extract_artifact(best_t),
                'rejected': extract_artifact(worst_t),
                'basis': f'self-play best-of-{RL2_K} ({best_s:.2f} vs {worst_s:.2f})',
            })
    print('self-play pairs:', len(selfplay_pairs))

    if len(selfplay_pairs) >= RL2_MIN_PAIRS:
        # 4b. short DPO round on the current policy
        free_memory(model)
        model = AutoModelForCausalLM.from_pretrained(
            GRPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
            attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
        )
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        )
        rl2_args = DPOConfig(
            output_dir=f'{CKPT}/rl2_run', run_name='prismacoder-3b-rl-round2-kaggle',
            seed=SEED, num_train_epochs=1, learning_rate=DPO_LR,
            lr_scheduler_type='cosine', warmup_ratio=0.03,
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            bf16=IS_BF16, fp16=not IS_BF16,
            gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
            optim='paged_adamw_8bit', max_grad_norm=1.0, beta=DPO_BETA,
            loss_type='sigmoid', max_prompt_length=1536, max_completion_length=1024,
            logging_steps=5, save_strategy='no', report_to=['tensorboard'],
        )
        rl2_trainer = DPOTrainer(
            model=model, args=rl2_args,
            train_dataset=Dataset.from_list([to_dpo_example(p, tokenizer) for p in selfplay_pairs]),
            processing_class=tokenizer,
            peft_config=LoraConfig(task_type='CAUSAL_LM', r=16, lora_alpha=32,
                                   lora_dropout=0.05, bias='none', target_modules=LORA_TARGETS),
        )
        rl2_trainer.train()
        rl2_trainer.save_model(f'{CKPT}/rl2_adapter')
        free_memory(rl2_trainer)

        # 4c. merge -> final model
        free_memory(model)
        base_cpu = AutoModelForCausalLM.from_pretrained(
            GRPO_MERGED, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True)
        peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/rl2_adapter')
        merged = peft_model.merge_and_unload()
        merged.save_pretrained(RL2_MERGED, safe_serialization=True)
        tokenizer.save_pretrained(RL2_MERGED)
        free_memory(base_cpu, peft_model, merged)

        if os.path.isdir(GRPO_MERGED) and GRPO_MERGED != RL2_MERGED:
            shutil.rmtree(GRPO_MERGED)
            print('freed disk:', GRPO_MERGED)

        FINAL_DIR = RL2_MERGED
        model = AutoModelForCausalLM.from_pretrained(
            FINAL_DIR, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
            attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
        )
        model.config.use_cache = False
        record_stage('rl_round2', gate_pass_rate(eval_pool))
    else:
        print(f'only {len(selfplay_pairs)} separable pairs (< {RL2_MIN_PAIRS}) '
              '- skipping the RL round; final model stays at', FINAL_DIR)
else:
    print('RUN_ITERATIVE_RL=False or empty pool - skipping')

In [ ]:
# Cell 21 - stage-wise progress: gate pass@1 table + chart
import json
import matplotlib
import matplotlib.pyplot as plt

results = load_stage_results()
order = [(k, results[k]) for k in ('base', 'sft', 'dpo', 'grpo', 'rl_round2') if k in results]
print('| Stage | V0 gate pass@1 |')
print('|---|---|')
for k, v in order:
    print(f'| {k} | {v:.3f} |')

if len(order) >= 2:
    labels = [k for k, _ in order]
    values = [v for _, v in order]
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    bars = ax.bar(labels, values, color=['#94a3b8', '#8b5cf6', '#6366f1', '#4f46e5', '#312e81'][:len(order)])
    ax.axhline(results.get('base', 0), color='#94a3b8', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('V0 gate pass@1 (held-out)')
    ax.set_title('PrismaCoder-3B: base -> SFT -> DPO -> GRPO -> RL round 2')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f'{v:.2f}',
                ha='center', va='bottom', fontsize=9)
    out_png = f'{WORK}/stage_progress.png'
    fig.savefig(out_png, dpi=150)
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    json.dump(dict(order), open(f'{WORK}/stage_results_final.json', 'w'), indent=2)
    print('chart ->', out_png)
    plt.show()

In [ ]:
# Cell 22 - push the final aligned model (+ stage results) to the HF Hub
import json
from huggingface_hub import HfApi

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_large_folder(repo_id=OUTPUT_REPO, repo_type='model', folder_path=FINAL_DIR)
    api.upload_file(
        path_or_fileobj=STAGE_RESULTS, path_in_repo='stage_results.json',
        repo_id=OUTPUT_REPO, repo_type='model')
    card = (
        '---\n'
        'license: apache-2.0\n'
        f'base_model: {BASE_MODEL}\n'
        'tags: [prisma, orm, text-to-code, qlora, dpo, grpo]\n'
        '---\n\n'
        '# PrismaCoder-3B-RL\n\n'
        'Qwen2.5-Coder-3B-Instruct aligned with the full pipeline: '
        'SFT -> DPO (verifier-labeled pairs) -> GRPO (verifier reward, online RL) '
        '-> iterative RL round. Data: PrismaForge. Benchmark: PrismaBench.\n\n'
        f'Stage-wise V0 gate pass@1: {json.dumps(load_stage_results())}\n'
    )
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo='README.md',
                    repo_id=OUTPUT_REPO, repo_type='model')
    print('pushed:', OUTPUT_REPO)
else:
    print('no HF token - final model kept at', FINAL_DIR)

In [ ]:
# Cell 23 - export GGUF (llama.cpp convert -> f16 -> q8_0)
import os, subprocess, sys

LC_DIR = '/kaggle/working/llama.cpp'
if not os.path.exists(LC_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggml-org/llama.cpp', LC_DIR], check=True)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    f'{LC_DIR}/requirements.txt'], check=False)
    f16 = f'{WORK}/PrismaCoder-3B-RL-f16.gguf'
    q8 = f'{WORK}/PrismaCoder-3B-RL-q8_0.gguf'
    subprocess.run([sys.executable, f'{LC_DIR}/convert_hf_to_gguf.py', FINAL_DIR,
                    '--outfile', f16, '--outtype', 'f16'], check=True)
    if os.path.exists(f'{LC_DIR}/build/bin/llama-quantize'):
        subprocess.run([f'{LC_DIR}/build/bin/llama-quantize', f16, q8, 'q8_0'], check=True)
    else:
        q8 = f16  # quantize binary not built - keep the f16 export
    if os.path.exists(q8) and q8 != f16 and os.path.exists(f16):
        os.remove(f16)                    # stay inside the /kaggle/working quota
    print('GGUF ->', q8)
except Exception as e:
    print('GGUF export failed (non-fatal):', type(e).__name__, '-', e)

## What you just built — and what to do next

`merged/rl2` (or `merged/grpo`) is the fully aligned **PrismaCoder-3B**:
`base → SFT (imitation) → DPO (verifier-labeled preferences) → GRPO (online RL,
verifier reward) → iterative RL round`, with every stage scored by the same gate
that filters PrismaForge and grades PrismaBench.

**Next steps**
- Full-gate evaluation: on a Node ≥ 18 machine run
  `python benchmark/evaluator/evaluate.py --tasks benchmark/public/prismabench_public.jsonl --submission <your generations> --model your-name --full-gates`
  — V0 passes here are `unverified` until the V1–V5 chain (prisma validate/generate,
  tsc, db push, vitest) re-runs them.
- Scale up: `MAX_TRAIN_SAMPLES=0`, `GRPO_PROMPTS=1000`, and the multi-GPU recipes in
  `training/configs/` (`full_ft.yaml`, `dpo_qlora.yaml`) reproduce the report's 120k-record run.
- Compare models pairwise: `evaluation/compare.py` (McNemar + bootstrap + Holm).

**Troubleshooting**
| Symptom | Fix |
|---|---|
| CUDA OOM in SFT/DPO | lower `per_device_train_batch_size` to 1 (raise `gradient_accumulation_steps` ×2) |
| GRPO slow | reduce `GRPO_PROMPTS` (e.g. 80) or `GRPO_MAX_COMPLETION` (e.g. 448); generation dominates |
| `num_generations` divisibility error | keep `per_device_train_batch_size × grad_accum` a multiple of `num_generations` (defaults: 2×4=8 ✓) |
| fp16 loss spike on T4 | set `SFT_EPOCHS=2`, keep `max_grad_norm=1.0`; T4 has no bf16 |
| Hub download 401/404 | expected before the dataset is published — the offline mini-corpus path runs automatically |
| Merges OOM (P100 13 GB RAM sessions) | use T4 ×2 (29 GB RAM) or merge with `torch_dtype=torch.float16` |

**Packages**: transformers 4.52.3 · trl 0.19.1 · peft 0.16.0 · bitsandbytes 0.46.1 —
pinned in Cell 1; bump as a set, not individually.
